# Stage 3 Test Runner

Exercise each stage 3 endpoint (entity, franchise, keyword, metadata, award, trending) in its own independently-runnable cell. Each endpoint cell runs generation (when applicable) followed by execution, printing the full LLM spec and the execution `EndpointResult`.

## Setup

Run this cell first. It imports everything each endpoint cell needs and defines shared LLM / input configuration. After this, any endpoint cell below can be run on its own.

In [1]:
import sys, os, time, asyncio, json
from datetime import date
from pathlib import Path

# Ensure project root is on the path so imports resolve.
project_root = str(Path.cwd().parent) if Path.cwd().name == "search_v2" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(Path(project_root) / ".env")

from implementation.llms.generic_methods import LLMProvider

# Stage 3 generation endpoints (LLM translators)
from search_v2.stage_3.entity_query_generation import generate_entity_query
from search_v2.stage_3.franchise_query_generation import generate_franchise_query
from search_v2.stage_3.keyword_query_generation import generate_keyword_query
from search_v2.stage_3.metadata_query_generation import generate_metadata_query
from search_v2.stage_3.award_query_generation import generate_award_query

# Stage 3 execution endpoints (DB / Redis scorers)
from search_v2.stage_3.entity_query_execution import execute_entity_query
from search_v2.stage_3.franchise_query_execution import execute_franchise_query
from search_v2.stage_3.keyword_query_execution import execute_keyword_query
from search_v2.stage_3.metadata_query_execution import execute_metadata_query
from search_v2.stage_3.award_query_execution import execute_award_query
from search_v2.stage_3.trending_query_execution import execute_trending_query
from search_v2.stage_3.semantic_query_generation import generate_semantic_preference_query, generate_semantic_dealbreaker_query

# ---- Database connections ----
# Mirrors the FastAPI lifespan handler in api/main.py: open the Postgres
# pool (created inert with open=False at import time), initialize the
# Redis async client, and ping Qdrant to fail fast if it's unreachable.
# Qdrant's AsyncQdrantClient is constructed at import time and needs no
# explicit open step — the ping here is just a readiness check.
from db.postgres import pool as postgres_pool
from db.redis import init_redis, check_redis
from db.qdrant import check_qdrant
import db.redis as _redis_module

# Idempotent: re-running the setup cell must not double-open the Postgres
# pool or leak a prior Redis client.
if postgres_pool._closed:
    await postgres_pool.open()
    print("Postgres: pool opened")
else:
    print("Postgres: pool already open")

if _redis_module._redis_client is None:
    await init_redis()
    print(f"Redis:    initialized ({await check_redis()})")
else:
    print(f"Redis:    already initialized ({await check_redis()})")

print(f"Qdrant:   {await check_qdrant()}")

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Postgres: pool opened
Redis:    initialized (ok)
Qdrant:   ok


## Configuration

Set the LLM provider/model, today's date, and the per-endpoint sample inputs. Each endpoint cell reads from this shared config so you can tweak inputs in one place.

In [2]:
# ---- Presets (uncomment one) ----
# Each preset sets provider, model, and kwargs for no-thinking mode.

# Kimi K2.5 — no thinking
# provider, model, kwargs = LLMProvider.KIMI, "kimi-k2.5", {"enable_thinking": False}

# GPT 5.4 Mini — no thinking
provider, model, kwargs = LLMProvider.OPENAI, "gpt-5.4-mini", {"reasoning_effort": "none", "verbosity":"low"}

# Gemini 3 Flash — no thinking
# provider, model, kwargs = LLMProvider.GEMINI, "gemini-3-flash-preview", {"thinking_config": {"thinking_budget": 0}}

# ---- Or set manually ----
# provider = LLMProvider.OPENAI
# model = "gpt-5-mini"
# kwargs = {"reasoning_effort": "low"}

# ---- Today's date (used by metadata + award generators for relative-year resolution) ----
today = date.today()

# ---- Per-endpoint sample inputs ----
# Each input block mirrors what stage 2 would hand to stage 3: an
# intent_rewrite (whole-query context), a description (the single
# positive-presence requirement), and a routing_rationale (the step 2
# concept-type label that routed this item here).

# Trending has no LLM — execution-only.

print(f"Provider: {provider.value}")
print(f"Model:    {model}")
print(f"Kwargs:   {kwargs or '(defaults)'}")
print(f"Today:    {today.isoformat()}")

Provider: openai
Model:    gpt-5.4-mini
Kwargs:   {'reasoning_effort': 'none', 'verbosity': 'low'}
Today:    2026-04-17


## Result Formatting Helper

`print_scored_candidates` takes an iterable of scored objects (anything with `movie_id` and `score` attributes — e.g. `ScoredCandidate`), bulk-fetches titles and release years from `movie_card` in one query, and prints each as `score  title (year)` on its own line. Each endpoint cell below calls it after printing the raw `EndpointResult`.

In [3]:
from datetime import datetime, timezone
from typing import Iterable, Protocol

from db.postgres import fetch_movie_cards


class _Scored(Protocol):
    movie_id: int
    score: float


async def print_scored_candidates(
    candidates: Iterable[_Scored],
    *,
    limit: int | None = 25,
    sort_desc: bool = True,
) -> None:
    """Print scored movies as ``score  title (year)``, one per line.

    Accepts any iterable of objects with ``movie_id`` and ``score``
    attributes — ``ScoredCandidate`` works, but so does any duck-typed
    equivalent. Bulk-fetches card metadata in a single query (per the
    "never query Postgres per-candidate" invariant in CLAUDE.md) and
    prints score first so the column aligns visually across rows.

    Args:
        candidates: Iterable of scored movies.
        limit: Max rows to print after sorting. None prints all.
        sort_desc: Sort by score descending before printing. Disable
            to preserve the caller's ordering.
    """
    items = list(candidates)
    if not items:
        print("(no scored candidates)")
        return

    if sort_desc:
        items = sorted(items, key=lambda c: c.score, reverse=True)
    if limit is not None:
        items = items[:limit]

    cards = await fetch_movie_cards([c.movie_id for c in items])
    card_by_id = {c["movie_id"]: c for c in cards}

    for cand in items:
        card = card_by_id.get(cand.movie_id)
        if card is None:
            # Movie exists in the scored set but not in movie_card — rare,
            # but possible for stale indexes or test data. Surface it
            # instead of silently skipping.
            title, year = "<missing card>", "?"
        else:
            title = card["title"] or "<untitled>"
            ts = card["release_ts"]
            year = (
                datetime.fromtimestamp(ts, tz=timezone.utc).year
                if ts is not None else "?"
            )
        print(f"  {cand.score:.3f}  {title} ({year})")

## Endpoint 1: Entity

Translates a named-entity lookup (person / character / studio / title pattern) into an `EntityQuerySpec` and executes it against the lexical posting tables.

In [24]:
entity_inputs = {
    "intent_rewrite": "Spider man movies",
    "description": "has spiderman character",
    "routing_rationale": "named person - character",
}

# ---- Generation ----
start = time.perf_counter()
entity_spec, entity_in_tok, entity_out_tok = await generate_entity_query(
    intent_rewrite=entity_inputs["intent_rewrite"],
    description=entity_inputs["description"],
    routing_rationale=entity_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
entity_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {entity_gen_elapsed:.2f}s")
print(f"Tokens — input: {entity_in_tok:,}  output: {entity_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — EntityQuerySpec")
print("=" * 60)
print(entity_spec.model_dump_json(indent=2))

# ---- Execution ----
start = time.perf_counter()
entity_result = await execute_entity_query(entity_spec)
entity_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT — {len(entity_result.scores)} movies in {entity_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(entity_result.scores)

Generation completed in 1.79s
Tokens — input: 3,044  output: 95

LLM RESULT — EntityQuerySpec
{
  "entity_type_evidence": "Character lookup: the description says a movie has the Spiderman character, and routing_rationale points to character.",
  "name_resolution_notes": "canonical character name",
  "lookup_text": "Spider-Man",
  "entity_type": "character",
  "person_category": null,
  "primary_category": null,
  "prominence_evidence": null,
  "actor_prominence_mode": null,
  "title_pattern_match_type": null,
  "character_alternative_names": [
    "Spiderman"
  ]
}

EXECUTION RESULT — 25 movies in 0.37s
  1.000  One Fine Day (1996)
  1.000  Spider-Man Strikes Back (1978)
  1.000  Captain America: Civil War (2016)
  1.000  Spider-Man: Across the Spider-Verse (2023)
  1.000  Venom: Let There Be Carnage (2021)
  1.000  The Amazing Spider-Man (2012)
  1.000  Don't Give a Damn (1995)
  1.000  Peter's To-Do List (2019)
  1.000  Avengers: Endgame (2019)
  1.000  Hollywoo (2011)
  1.000  Aveng

## Endpoint 2: Franchise

Translates a franchise / shared-universe requirement into a `FranchiseQuerySpec` and executes it against `movie_franchise_metadata`.

In [ ]:
franchise_inputs = {
    "intent_rewrite": "Find Marvel Cinematic Universe movies",
    "description": "part of the Marvel Cinematic Universe",
    "routing_rationale": "named franchise / shared universe",
}

# ---- Generation ----
start = time.perf_counter()
franchise_spec, franchise_in_tok, franchise_out_tok = await generate_franchise_query(
    intent_rewrite=franchise_inputs["intent_rewrite"],
    description=franchise_inputs["description"],
    routing_rationale=franchise_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
franchise_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {franchise_gen_elapsed:.2f}s")
print(f"Tokens — input: {franchise_in_tok:,}  output: {franchise_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — FranchiseQuerySpec")
print("=" * 60)
print(franchise_spec.model_dump_json(indent=2))

# ---- Execution ----
start = time.perf_counter()
franchise_result = await execute_franchise_query(franchise_spec)
franchise_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT — {len(franchise_result.scores)} movies in {franchise_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(franchise_result.scores)

Generation completed in 1.47s
Tokens — input: 1,987  output: 109

LLM RESULT — FranchiseQuerySpec
{
  "concept_analysis": "The description 'part of the Marvel Cinematic Universe' explicitly identifies a named cinematic universe (lineage_or_universe_names). No named subgroup, lineage position (sequel/prequel), structural flags (spinoff/crossover), or launch behavior (started a franchise) is mentioned in the description.",
  "lineage_or_universe_names": [
    "marvel cinematic universe",
    "marvel"
  ],
  "recognized_subgroups": null,
  "lineage_position": null,
  "structural_flags": null,
  "launch_scope": null
}

EXECUTION RESULT — 51 movies in 0.28s
  1.000  Guardians of the Galaxy Vol. 3 (2023)
  1.000  Captain America: Civil War (2016)
  1.000  Peter's To-Do List (2019)
  1.000  Avengers: Endgame (2019)
  1.000  Avengers: Infinity War (2018)
  1.000  Marvel One-Shot: Item 47 (2012)
  1.000  Captain Marvel (2019)
  1.000  Doctor Strange in the Multiverse of Madness (2022)
  1.000  

## Endpoint 3: Keyword

Translates a classification / tag requirement into a `KeywordQuerySpec` (one `UnifiedClassification` registry pick) and executes the backing posting-list lookup.

In [ ]:
keyword_inputs = {
    "intent_rewrite": "Find animated movies for kids",
    "description": "is an animated movie",
    "routing_rationale": "genre classification",
}

# ---- Generation ----
start = time.perf_counter()
keyword_spec, keyword_in_tok, keyword_out_tok = await generate_keyword_query(
    intent_rewrite=keyword_inputs["intent_rewrite"],
    description=keyword_inputs["description"],
    routing_rationale=keyword_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
keyword_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {keyword_gen_elapsed:.2f}s")
print(f"Tokens — input: {keyword_in_tok:,}  output: {keyword_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — KeywordQuerySpec")
print("=" * 60)
print(keyword_spec.model_dump_json(indent=2))

# ---- Execution ----
start = time.perf_counter()
keyword_result = await execute_keyword_query(keyword_spec)
keyword_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT — {len(keyword_result.scores)} movies in {keyword_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(keyword_result.scores)

Generation completed in 3.86s
Tokens — input: 10,865  output: 69

LLM RESULT — KeywordQuerySpec
{
  "concept_analysis": "\"animated\" → Animation family; \"for kids\" → audience suitability, but the explicit requirement is form/medium, not family tag",
  "candidate_shortlist": "ANIMATION: animated form — present | FAMILY: made for both children and adults — not stated",
  "classification": "ANIMATION"
}

EXECUTION RESULT — 6009 movies in 4.98s
  1.000  Trapito (1975)
  1.000  The Simpsons Movie (2007)
  1.000  In the Blue Sea, in the White Foam... (1984)
  1.000  The Milky Way (1940)
  1.000  How to Ride a Horse (1947)
  1.000  Princes and Princesses (2000)
  1.000  Little Gobie (2010)
  1.000  The Snail and the Whale (2020)
  1.000  Psycho-Pass 3: First Inspector (2020)
  1.000  The Donkey King (2018)
  1.000  Poupelle of Chimney Town (2020)
  1.000  Nausicaä of the Valley of the Wind (1984)
  1.000  Mobile Suit Gundam SEED FREEDOM (2024)
  1.000  Lupin the Third: The Legend of the Go

## Endpoint 4: Metadata

Translates a structured-attribute requirement (date, runtime, maturity, streaming, language, country, budget, box office, popularity, reception) into a `MetadataTranslationOutput` and executes the scoring query against `movie_card`.

In [ ]:
metadata_inputs = {
    "intent_rewrite": "Find recent movies released in the last couple of years",
    "description": "released in the last 2 years",
    "routing_rationale": "release date / temporal attribute",
}

# ---- Generation ----
start = time.perf_counter()
metadata_spec, metadata_in_tok, metadata_out_tok = await generate_metadata_query(
    intent_rewrite=metadata_inputs["intent_rewrite"],
    description=metadata_inputs["description"],
    routing_rationale=metadata_inputs["routing_rationale"],
    today=today,
    provider=provider,
    model=model,
    **kwargs,
)
metadata_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {metadata_gen_elapsed:.2f}s")
print(f"Tokens — input: {metadata_in_tok:,}  output: {metadata_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — MetadataTranslationOutput")
print("=" * 60)
print(metadata_spec.model_dump_json(indent=2))

# ---- Execution ----
start = time.perf_counter()
metadata_result = await execute_metadata_query(metadata_spec)
metadata_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT — {len(metadata_result.scores)} movies in {metadata_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(metadata_result.scores)

Generation completed in 2.18s
Tokens — input: 6,679  output: 122

LLM RESULT — MetadataTranslationOutput
{
  "constraint_phrases": [
    "recent",
    "released in the last 2 years",
    "last couple of years"
  ],
  "target_attribute": "release_date",
  "value_intent_label": "last 2 years",
  "release_date": {
    "first_date": "2024-04-17",
    "match_operation": "between",
    "second_date": "2026-04-17"
  },
  "runtime": null,
  "maturity_rating": null,
  "streaming": null,
  "audio_language": null,
  "country_of_origin": null,
  "budget_scale": null,
  "box_office": null,
  "popularity": null,
  "reception": null
}

EXECUTION RESULT — 8981 movies in 2.60s
  1.000  Trending (2025)
  1.000  Virus (2025)
  1.000  Make a Wish (2024)
  1.000  Queen at Sea (2026)
  1.000  Decibel (2024)
  1.000  Mananambal (2025)
  1.000  Coffintooth (2024)
  1.000  Innocent (2025)
  1.000  Eat, Love, London (2024)
  1.000  Hunting Daze (2024)
  1.000  An Angry Boy (2024)
  1.000  Murder at Hollow Creek

## Endpoint 5: Award

Translates an award / prestige requirement into an `AwardQuerySpec` and executes it against `movie_awards` (with the `award_ceremony_win_ids` fast path when applicable).

In [20]:
award_inputs = {
    "intent_rewrite": "Find movies that won the Oscar for Best Picture this decade",
    "description": "won the Academy Award for Best Picture this decade",
    "routing_rationale": "award / prestige recognition",
}

# ---- Generation ----
start = time.perf_counter()
award_spec, award_in_tok, award_out_tok = await generate_award_query(
    intent_rewrite=award_inputs["intent_rewrite"],
    description=award_inputs["description"],
    routing_rationale=award_inputs["routing_rationale"],
    today=today,
    provider=provider,
    model=model,
    **kwargs,
)
award_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {award_gen_elapsed:.2f}s")
print(f"Tokens — input: {award_in_tok:,}  output: {award_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — AwardQuerySpec")
print("=" * 60)
print(award_spec.model_dump_json(indent=2))

# ---- Execution ----
start = time.perf_counter()
award_result = await execute_award_query(award_spec)
award_exec_elapsed = time.perf_counter() - start

print()
print("=" * 60)
print(f"EXECUTION RESULT — {len(award_result.scores)} movies in {award_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(award_result.scores)

Generation completed in 2.38s
Tokens — input: 6,517  output: 164

LLM RESULT — AwardQuerySpec
{
  "concept_analysis": "ceremony: \"Academy Award\" signals Academy Awards, USA; award name: \"Best Picture\" signals the Oscar prize context but the direct named prize surface is \"Oscar\"; category: \"Best Picture\" signals best-picture; outcome: \"won\" signals winner; year: \"this decade\" signals a year range based on today (2026-04-17), i.e. 2020-2026.",
  "scoring_shape_label": "specific filter, no count",
  "scoring_mode": "floor",
  "scoring_mark": 1,
  "ceremonies": [
    "Academy Awards, USA"
  ],
  "award_names": [
    "Oscar"
  ],
  "category_tags": [
    "best-picture"
  ],
  "outcome": "winner",
  "years": {
    "year_from": 2020,
    "year_to": 2026
  }
}

EXECUTION RESULT — 7 movies in 0.10s
  1.000  Parasite (2019)
  1.000  Everything Everywhere All at Once (2022)
  1.000  Nomadland (2021)
  1.000  CODA (2021)
  1.000  Oppenheimer (2023)
  1.000  One Battle After Another (20

## Endpoint 6: Trending

Trending has no LLM translator — stage 2 flags the intent and execution reads the precomputed trending hash from Redis directly.

In [21]:
# ---- Execution only (no generation step) ----
start = time.perf_counter()
trending_result = await execute_trending_query()
trending_exec_elapsed = time.perf_counter() - start

print("=" * 60)
print("LLM RESULT — (none; trending has no LLM translator)")
print("=" * 60)
print()
print("=" * 60)
print(f"EXECUTION RESULT — {len(trending_result.scores)} movies in {trending_exec_elapsed:.2f}s")
print("=" * 60)
await print_scored_candidates(trending_result.scores)

Redis key 'dev:trending:current' is absent or empty — trending scores unavailable


LLM RESULT — (none; trending has no LLM translator)

EXECUTION RESULT — 0 movies in 0.13s
(no scored candidates)


## Endpoint 7a: Semantic — Dealbreaker Flow

Translates a single positive-presence semantic trait into a `SemanticDealbreakerSpec` — one LLM call picks exactly one of 7 non-anchor vector spaces and emits a structured-label body. Generation only; no execution endpoint exists yet.

In [4]:
semantic_dealbreaker_inputs = {
    "intent_rewrite": "Find slow-burn psychological thrillers with an unreliable narrator",
    "description": "features an unreliable narrator as a central storytelling device",
    "routing_rationale": "narrative craft / storytelling technique",
}

# ---- Generation ----
start = time.perf_counter()
semantic_db_spec, semantic_db_in_tok, semantic_db_out_tok = await generate_semantic_dealbreaker_query(
    intent_rewrite=semantic_dealbreaker_inputs["intent_rewrite"],
    description=semantic_dealbreaker_inputs["description"],
    routing_rationale=semantic_dealbreaker_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
semantic_db_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {semantic_db_gen_elapsed:.2f}s")
print(f"Tokens — input: {semantic_db_in_tok:,}  output: {semantic_db_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — SemanticDealbreakerSpec")
print("=" * 60)
print(semantic_db_spec.model_dump_json(indent=2))

Generation completed in 4.06s
Tokens — input: 4,398  output: 211

LLM RESULT — SemanticDealbreakerSpec
{
  "signal_inventory": "\"unreliable narrator\" → narrative_techniques (pov_perspective, information_control, additional_narrative_devices); \"central storytelling device\" → narrative_techniques (narrative_delivery, information_control); no phrase implicates plot_events, plot_analysis, viewer_experience, watch_context, production, or reception",
  "target_fields_label": "pov_perspective, information_control, additional_narrative_devices",
  "body": {
    "space": "narrative_techniques",
    "content": {
      "narrative_archetype": {
        "terms": []
      },
      "narrative_delivery": {
        "terms": []
      },
      "pov_perspective": {
        "terms": [
          "unreliable narrator"
        ]
      },
      "characterization_methods": {
        "terms": []
      },
      "character_arcs": {
        "terms": []
      },
      "audience_character_perception": {
        "

## Endpoint 7b: Semantic — Preference Flow

Translates a grouped preference description into a `SemanticPreferenceSpec` — decomposes into atomic qualifiers, picks 1+ of the 8 vector spaces (anchor allowed), assigns primary/contributing weights, and emits one body per selected space. Generation only; no execution endpoint exists yet.

In [5]:
semantic_preference_inputs = {
    "intent_rewrite": "Something slow-burn, atmospheric, and melancholy for a rainy Sunday afternoon",
    "description": "slow-burn, atmospheric, rainy-day melancholy for a quiet Sunday afternoon",
    "routing_rationale": "viewing mood / tone preference",
}

# ---- Generation ----
start = time.perf_counter()
semantic_pref_spec, semantic_pref_in_tok, semantic_pref_out_tok = await generate_semantic_preference_query(
    intent_rewrite=semantic_preference_inputs["intent_rewrite"],
    description=semantic_preference_inputs["description"],
    routing_rationale=semantic_preference_inputs["routing_rationale"],
    provider=provider,
    model=model,
    **kwargs,
)
semantic_pref_gen_elapsed = time.perf_counter() - start

print(f"Generation completed in {semantic_pref_gen_elapsed:.2f}s")
print(f"Tokens — input: {semantic_pref_in_tok:,}  output: {semantic_pref_out_tok:,}\n")
print("=" * 60)
print("LLM RESULT — SemanticPreferenceSpec")
print("=" * 60)
print(semantic_pref_spec.model_dump_json(indent=2))

Generation completed in 4.69s
Tokens — input: 5,923  output: 463

LLM RESULT — SemanticPreferenceSpec
{
  "qualifier_inventory": "\"slow-burn\" → viewer_experience + plot_events; \"atmospheric\" → viewer_experience + anchor; \"melancholy\" → viewer_experience + anchor; \"rainy-day\" → watch_context + anchor; \"quiet Sunday afternoon\" → watch_context + anchor",
  "space_queries": [
    {
      "carries_qualifiers": "carries: slow-burn, atmospheric, melancholy",
      "space": "viewer_experience",
      "weight": "primary",
      "content": {
        "emotional_palette": {
          "terms": [
            "melancholy",
            "reflective",
            "subdued"
          ],
          "negations": []
        },
        "tension_adrenaline": {
          "terms": [
            "slow-burn",
            "restrained",
            "simmering"
          ],
          "negations": []
        },
        "tone_self_seriousness": {
          "terms": [
            "atmospheric",
            "qu